[![cloudevel](img/cloudevel.png)](https://cloudevel.com)

# El procesador de texto *gawk*.

## Antecedentes.

*awk* fue creado en 1977 por Alfred **A**ho, Peter **W**einberger y Brian **K**ernighan en los laboratorios Bell. A diferencia de *sed* y *vim*, *awk* no desciende de *ed*: es un lenguaje de programación diseñado desde cero para el procesamiento de texto estructurado en columnas.

*gawk* es la implementación *GNU* de *awk* y es la versión estándar en la mayoría de las distribuciones *GNU/Linux*.

```
ed  (1969) → sed, vi/vim
awk (1977) → gawk          ← origen independiente
```

Mientras *sed* transforma líneas de texto, *gawk* trata cada línea como un **registro** dividido en **campos**, lo que lo hace ideal para procesar archivos estructurados como *CSV*, logs, salidas de comandos, etc.

## Sintaxis básica.

```bash
gawk 'patrón { acción }' archivo
```

O recibiendo entrada de un pipe:

```bash
comando | gawk 'patrón { acción }'
```

* Si se omite el **patrón**, la acción se aplica a todas las líneas.
* Si se omite la **acción**, se imprime la línea completa (`print $0`).
* Pueden encadenarse múltiples reglas `patrón { acción }`.

## Variables predefinidas.

| Variable | Descripción |
|----------|-------------|
| `$0`     | Línea completa (registro actual) |
| `$1`, `$2`, …, `$NF` | Campos 1, 2, … hasta el último |
| `NF`     | Número de campos en el registro actual |
| `NR`     | Número de registro (línea) actual |
| `FS`     | Separador de campos de entrada (por defecto: espacio/tab) |
| `OFS`    | Separador de campos de salida (por defecto: espacio) |
| `RS`     | Separador de registros de entrada (por defecto: salto de línea) |
| `ORS`    | Separador de registros de salida (por defecto: salto de línea) |
| `FILENAME` | Nombre del archivo que se está procesando |

In [ ]:
echo "uno dos tres" | gawk '{ print $2 }'

In [ ]:
echo "uno dos tres" | gawk '{ print NF, $NF }'

In [ ]:
printf 'a b c\nd e f\n' | gawk '{ print NR, $0 }'

## Separador de campos con ```-F```.

La opción `-F` define el separador de campos de entrada. Puede ser un carácter o una expresión regular.

In [ ]:
echo "root:x:0:0:root:/root:/bin/bash" | gawk -F: '{ print $1, $6 }'

In [ ]:
gawk -F: '{ print $1, $3 }' /etc/passwd | head -5

In [ ]:
echo "2026-02-21" | gawk -F- '{ print "año:" $1, "mes:" $2, "día:" $3 }'

## Patrones.

Los patrones filtran qué líneas reciben la acción. Pueden ser:

* **Expresiones regulares**: `/regex/`
* **Comparaciones**: `$3 > 100`, `$1 == "root"`
* **Rangos**: `/inicio/, /fin/`

In [ ]:
gawk -F: '/bash$/ { print $1 }' /etc/passwd

In [ ]:
gawk -F: '$3 >= 1000 { print $1, $3 }' /etc/passwd

## Los bloques ```BEGIN``` y ```END```.

* El bloque `BEGIN` se ejecuta **antes** de procesar cualquier línea: útil para inicializar variables, imprimir cabeceras o definir el separador de campos (`FS`).
* El bloque `END` se ejecuta **después** de procesar todas las líneas: útil para imprimir totales o resúmenes.

In [ ]:
gawk 'BEGIN { print "=== Usuarios del sistema ===" }
     /bash$/ { print $1 }
     END   { print "=== Fin ==="}' FS=: /etc/passwd

In [ ]:
gawk -F: 'BEGIN { total=0 }
          $3 >= 1000 { total++ }
          END { print "Usuarios normales:", total }' /etc/passwd

## Operaciones aritméticas y acumuladores.

In [ ]:
df -k | gawk 'NR > 1 && $3 ~ /^[0-9]/ { sum += $3 }
              END { printf "Espacio usado total: %.1f MB\n", sum/1024 }'

## El comando ```printf``` en *gawk*.

*gawk* incluye `printf` con la misma sintaxis que C, lo que permite formatear la salida con precisión.

In [ ]:
gawk -F: '$3 < 100 { printf "%-15s UID: %4d\n", $1, $3 }' /etc/passwd | head -8

## Procesamiento de logs.

*gawk* es especialmente útil para analizar archivos de log. El siguiente ejemplo extrae información del log de autenticación del sistema.

In [ ]:
sudo gawk '/Failed password/ { print $1, $2, $3, "→", $11 }' /var/log/auth.log 2>/dev/null | tail -5

## La man page de ```gawk```.

In [ ]:
man gawk

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>